In [1]:
import pandas as pd
import numpy as np
import os
from scipy.stats import skew, kurtosis
from astropy.timeseries import LombScargle

# 1. Load the processed static feature matrix
input_path = '../data/processed/feature_matrix.csv'
if not os.path.exists(input_path):
    raise FileNotFoundError(f"Feature matrix not found at {input_path}. Run notebook 04 first.")

static_df = pd.read_csv(input_path)

# 2. Define the Temporal Feature Extraction Function
def extract_lightcurve_features(time, flux):
    """
    Extracts statistical and periodic features from a single astronomical light curve.
    """
    features = {}
    
    # Basic statistics
    features['lc_mean'] = np.mean(flux)
    features['lc_std'] = np.std(flux)
    features['lc_amplitude'] = (np.max(flux) - np.min(flux)) / 2.0
    features['lc_skew'] = skew(flux)
    features['lc_kurtosis'] = kurtosis(flux)
    
    # Periodicity estimation using Astropy's Lomb-Scargle periodogram
    # We test frequencies corresponding to periods between 0.1 and 100 days
    frequency, power = LombScargle(time, flux).autopower(minimum_frequency=0.01, maximum_frequency=10.0)
    best_frequency = frequency[np.argmax(power)]
    features['lc_period'] = 1.0 / best_frequency
    features['lc_max_power'] = np.max(power) # Confidence in the period
    
    return features

# 3. Simulate Light Curve Data (Placeholder for Gaia DataLink API)
print("Processing light curves and extracting temporal features...")
temporal_features_list = []

# For demonstration, we iterate over the first 1000 objects to save time.
# In production, you will query the Gaia DataLink service for each source_id.
np.random.seed(42)
for index, row in static_df.head(1000).iterrows():
    source_id = row['source_id']
    
    # Simulate a light curve: 50 observations over 100 days
    time = np.sort(np.random.uniform(0, 100, 50))
    # Base flux + some periodic signal + noise
    true_period = np.random.uniform(1, 20)
    flux = 15.0 + 0.5 * np.sin(2 * np.pi * time / true_period) + np.random.normal(0, 0.1, 50)
    
    # Extract features
    extracted = extract_lightcurve_features(time, flux)
    extracted['source_id'] = source_id
    temporal_features_list.append(extracted)

temporal_df = pd.DataFrame(temporal_features_list)

# 4. Merge Static (Astrometry/Photometry) and Temporal Features
# We use a left join to keep all original objects, even if they lacked temporal data
multimodal_df = pd.merge(static_df, temporal_df, on='source_id', how='left')

# 5. Handle missing temporal data (impute with median for standard ML pipelines)
temporal_cols = ['lc_mean', 'lc_std', 'lc_amplitude', 'lc_skew', 'lc_kurtosis', 'lc_period', 'lc_max_power']
for col in temporal_cols:
    multimodal_df[col] = multimodal_df[col].fillna(multimodal_df[col].median())

# 6. Save the Multimodal Feature Matrix
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'multimodal_feature_matrix.csv')
multimodal_df.to_csv(output_path, index=False)

print(f"Temporal feature extraction complete. Multimodal matrix saved to {output_path}")
display(multimodal_df[['source_id', 'proper_motion_total', 'lc_period', 'lc_amplitude']].head())

Processing light curves and extracting temporal features...
Temporal feature extraction complete. Multimodal matrix saved to ../data/processed/multimodal_feature_matrix.csv


,source_id,proper_motion_total,lc_period,lc_amplitude
0,137342093370833408,5.264533,19.180904,0.560990
1,137346392632994944,12.166480,1.315831,0.608905
2,137346564431685120,31.598544,11.922266,0.700723
3,137349519369183744,0.947780,18.098139,0.603146
4,137349691167876864,6.790614,8.052094,0.604994
